In [1]:
# ============================================================
#  FINE-TUNING A MODEL TO ANSWER IN LEGAL TERMS
#  Easy classroom demo — runs on Google Colab (free GPU)
#  Library: Hugging Face Transformers + PEFT (LoRA)
# ============================================================

# ── STEP 0: Install libraries ────────────────────────────────
# Run this cell first, then restart the runtime when asked

# !pip install -q transformers datasets peft accelerate bitsandbytes


# ── STEP 1: Imports ──────────────────────────────────────────

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import torch

print("✅ Libraries loaded")


# ── STEP 2: Small Legal Q&A Dataset ─────────────────────────
# Each example is a prompt → legal-style answer pair.
# In class: explain that the model will LEARN this style.

raw_data = [
    {
        "prompt": "What happens if I break a contract?",
        "response": (
            "A breach of contract occurs when a party fails to fulfill their "
            "contractual obligations without lawful excuse. The non-breaching party "
            "may be entitled to remedies including compensatory damages, specific "
            "performance, or rescission of the agreement, subject to mitigation duties."
        ),
    },
    {
        "prompt": "Can my landlord evict me without notice?",
        "response": (
            "Pursuant to landlord-tenant statutes, a lessor is generally required to "
            "provide adequate written notice prior to initiating eviction proceedings. "
            "Failure to comply with statutory notice requirements may render the "
            "eviction unlawful and constitute wrongful eviction."
        ),
    },
    {
        "prompt": "What is negligence?",
        "response": (
            "Negligence is a tort arising when a party breaches their duty of care "
            "owed to another, proximately causing foreseeable harm or damages. "
            "The claimant must establish four elements: duty, breach, causation, "
            "and resultant damages to sustain a negligence claim."
        ),
    },
    {
        "prompt": "What is intellectual property?",
        "response": (
            "Intellectual property refers to legally recognized exclusive rights "
            "granted to creators over their intangible creations, encompassing "
            "patents, trademarks, copyrights, and trade secrets. These rights "
            "enable rights-holders to exercise control over use, reproduction, "
            "and commercialization of their protected works."
        ),
    },
    {
        "prompt": "What does 'beyond reasonable doubt' mean?",
        "response": (
            "Beyond reasonable doubt is the highest standard of proof in common law "
            "jurisdictions, required to sustain a criminal conviction. It mandates "
            "that the trier of fact be firmly convinced of the accused's guilt such "
            "that no reasonable alternative hypothesis consistent with innocence exists."
        ),
    },
    {
        "prompt": "Can I sue someone for defamation?",
        "response": (
            "Defamation is a tort involving the publication of a false statement of "
            "fact that injures the reputation of an identifiable person. A claimant "
            "must establish that the statement was false, defamatory, published to a "
            "third party, and caused reputational or pecuniary harm, subject to "
            "applicable privileges and defences."
        ),
    },
    {
        "prompt": "What is a power of attorney?",
        "response": (
            "A power of attorney is a legal instrument by which a principal confers "
            "authority upon an agent, known as the attorney-in-fact, to act on their "
            "behalf in specified legal, financial, or personal matters. The scope of "
            "authority may be general or limited, and terminates upon the principal's "
            "death or revocation, unless constituted as a durable power of attorney."
        ),
    },
    {
        "prompt": "What is habeas corpus?",
        "response": (
            "Habeas corpus is a fundamental writ of common law origin compelling a "
            "custodian to produce a detained person before the court and justify the "
            "lawfulness of their detention. It serves as a critical safeguard against "
            "unlawful imprisonment and is a cornerstone of individual liberty "
            "protections in constitutional democracies."
        ),
    },
]

print(f"✅ Dataset ready — {len(raw_data)} examples")




✅ Libraries loaded
✅ Dataset ready — 8 examples


In [6]:


# ── STEP 3: Format as instruction text ───────────────────────
# We combine prompt + response into a single training string.
# The model learns: "when I see this prompt style → answer legally"

def format_example(example):
    text = (
        f"### Question:\n{example['prompt']}\n\n"
        f"### Legal Answer:\n{example['response']}"
    )
    return {"text": text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_example)
print("✅ Dataset formatted")
print("\nSample training text:\n")
print(dataset[0]["text"])


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

✅ Dataset formatted

Sample training text:

### Question:
What happens if I break a contract?

### Legal Answer:
A breach of contract occurs when a party fails to fulfill their contractual obligations without lawful excuse. The non-breaching party may be entitled to remedies including compensatory damages, specific performance, or rescission of the agreement, subject to mitigation duties.


In [7]:


# ── STEP 4: Load Tokenizer & Model ───────────────────────────
# We use a small model so it fits on Colab's free GPU.
# GPT-2 is tiny, well-known, and perfect for teaching.

MODEL_NAME = "gpt2"   # 117M params — fast, free, educational

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print(f"✅ Model loaded: {MODEL_NAME}")


# ── STEP 5: Apply LoRA (PEFT) ────────────────────────────────
# LoRA = Low-Rank Adaptation.
# Instead of updating ALL weights, we add small trainable matrices.
#
# TEACH THIS: "We freeze 99% of the model. We only train tiny adapters."
#             "This is why fine-tuning is fast and cheap."

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # We're fine-tuning a text-generation model
    r=8,                           # Rank of the adapter matrices (keep small)
    lora_alpha=32,                 # Scaling factor
    lora_dropout=0.1,              # Dropout for regularization
    target_modules=["c_attn"],     # Which layers to apply LoRA to (attention layers)
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # Shows how few params we actually train!


# ── STEP 6: Tokenize the Dataset ─────────────────────────────

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)
print("✅ Dataset tokenized")


# ── STEP 7: Training Arguments ───────────────────────────────
# These control HOW the training runs.
# Keep epochs low for a classroom demo — fast but visible results.

training_args = TrainingArguments(
    output_dir="./legal-finetuned-model",  # Where to save checkpoints
    num_train_epochs=10,                   # How many times to loop over data
    per_device_train_batch_size=2,         # Examples per step
    learning_rate=2e-4,                    # Step size for gradient updates
    logging_steps=5,                       # Print loss every N steps
    save_strategy="no",                    # Don't save during demo (saves time)
    report_to="none",                      # No external logging (wandb etc.)
    fp16=torch.cuda.is_available(),        # Use half-precision if GPU available
)


# ── STEP 8: Train! ───────────────────────────────────────────

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False   # mlm=False → causal LM (not masked)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("🚀 Starting fine-tuning...\n")
trainer.train()
print("\n✅ Fine-tuning complete!")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded: gpt2
trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

✅ Dataset tokenized
🚀 Starting fine-tuning...



Step,Training Loss
5,3.602163
10,3.710100
15,3.446427
20,3.356804
25,3.284209
30,3.373714
35,3.290724
40,3.284631



✅ Fine-tuning complete!


In [4]:


# ── STEP 9: Test the Fine-Tuned Model ────────────────────────
# Ask it a legal question and see if it answers in legal style.

def ask_legal(question, max_new_tokens=120):
    prompt = f"### Question:\n{question}\n\n### Legal Answer:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {question}")
    print(f"A: {generated.strip()}\n")

# Try these questions:
ask_legal("What is a contract?")
# ask_legal("What happens if someone doesn't pay me?")
# ask_legal("Can the police search my house?")


# ── STEP 10 (Optional): Save & Reload ────────────────────────
# model.save_pretrained("./my-legal-model")
# tokenizer.save_pretrained("./my-legal-model")
# Then reload: model = AutoModelForCausalLM.from_pretrained("./my-legal-model")


# ============================================================
#  CLASSROOM DISCUSSION POINTS
# ============================================================
#
#  1. WHAT DID WE FINE-TUNE?
#     → GPT-2, a causal language model (predicts next token)
#
#  2. WHAT IS LoRA?
#     → Instead of retraining all 117M weights, we add tiny
#       "adapter" matrices to just the attention layers.
#       Only ~0.5% of parameters are actually trained!
#
#  3. WHY IS THE DATASET SO SMALL?
#     → Fine-tuning only adjusts the *style*, not the knowledge.
#       GPT-2 already knows English — we're teaching it to
#       *write like a lawyer*.
#
#  4. WHAT WOULD MAKE IT BETTER?
#     → More examples, a larger model (GPT-2 medium/large),
#       more epochs, or a model already trained on legal text.
#
#  5. REAL-WORLD FINE-TUNING:
#     → Same concept, but with models like LLaMA-3, Mistral,
#       or Phi-3 and thousands of examples.
# ============================================================

Q: What is a contract?
A: A contract is an agreement between a company and its representatives, who may or may act as agents to protect and defend a given company against a particular risk. The terms of the agreement can be varied between the parties, depending on the circumstances. The terms of the contract can be defined by the parties and can be varied by the parties. The parties, and the parties themselves, may change the terms of a contract at any time.

### How does a contract affect a customer?

A contract is a contract between the parties that will result in the agreement, when the parties have agreed.

